# Buổi 4 — Vector database với Chroma

Ta tách rõ hai vai trò: Sentence Transformers tạo embedding; Chroma lưu vector cùng ID, văn bản và metadata rồi tìm lân cận.

## Chuẩn bị

~~~bash
python -m pip install -U sentence-transformers chromadb
~~~

Notebook tạo thư mục chroma_data cạnh file này. Chạy từng cell từ trên xuống.

## 1. Corpus, ID và metadata

Metadata cho biết đoạn đến từ chủ đề, file hoặc trang nào. Trong dự án thật, source có thể là tên file hoặc URL.

In [ ]:
from pathlib import Path
import chromadb
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)
ids = ["p1", "p2", "p3", "p4", "p5", "p6"]
docs = [
    "TF-IDF giảm trọng số của từ phổ biến và tăng trọng số của từ đặc trưng.",
    "Embedding biến câu thành vector để so sánh ý nghĩa.",
    "Naive Bayes có thể phân loại email spam dựa trên xác suất từ.",
    "Chroma lưu vector, ID, nội dung và metadata trong collection.",
    "Truy vấn vector tìm những đoạn gần nhất theo metric đã chọn.",
    "Metadata giúp lọc kết quả theo chủ đề hoặc nguồn tài liệu."
]
metadatas = [
    {"topic": "tfidf", "source": "buoi_2.pdf", "page": 1},
    {"topic": "embedding", "source": "buoi_3.md", "page": 1},
    {"topic": "classification", "source": "buoi_1.ipynb", "page": 2},
    {"topic": "vector_db", "source": "buoi_4.md", "page": 1},
    {"topic": "retrieval", "source": "buoi_4.md", "page": 2},
    {"topic": "retrieval", "source": "buoi_4.md", "page": 3}
]
embeddings = model.encode(
    docs, convert_to_numpy=True, normalize_embeddings=True
)
print("Số vector:", len(embeddings), "| số chiều:", embeddings.shape[1])

## 2. Tạo collection và ghi dữ liệu

PersistentClient lưu database cục bộ. Metric cosine phù hợp với vector đã chuẩn hóa. Upsert giúp chạy lại cell với cùng ID mà không tạo bản ghi trùng.

In [ ]:
database_path = Path("chroma_data").resolve()
client = chromadb.PersistentClient(path=str(database_path))
collection = client.get_or_create_collection(
    name="vietnamese_nlp_lessons",
    metadata={"hnsw:space": "cosine"}
)
collection.upsert(
    ids=ids,
    documents=docs,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)
print("Đường dẫn dữ liệu:", database_path)
print("Số bản ghi:", collection.count())

## 3. Truy vấn top-k

Chroma trả về distance. Với cosine distance, giá trị nhỏ hơn nghĩa là gần hơn; đây không phải cosine similarity.

In [ ]:
query = "Làm thế nào lưu vector để tìm kiếm theo ý nghĩa?"
query_embedding = model.encode(
    [query], convert_to_numpy=True, normalize_embeddings=True
)[0]
results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)
for doc_id, text, metadata, distance in zip(
    results["ids"][0],
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):
    print(f"{doc_id} | cosine distance={distance:.4f} | {metadata} | {text}")

## 4. Lọc metadata và persistence

Điều kiện where lọc theo metadata. Tạo client mới để xác nhận collection đã được lưu xuống đĩa.

In [ ]:
filtered = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=3,
    where={"topic": "retrieval"},
    include=["documents", "metadatas", "distances"]
)
print("Chỉ các bản ghi topic=retrieval:")
for doc_id, text, metadata in zip(
    filtered["ids"][0],
    filtered["documents"][0],
    filtered["metadatas"][0]
):
    print(doc_id, metadata, text)

client_reopened = chromadb.PersistentClient(path=str(database_path))
collection_reopened = client_reopened.get_collection("vietnamese_nlp_lessons")
print("Số bản ghi sau khi mở lại:", collection_reopened.count())

## Bài tập

1. Thêm metadata source, topic và page cho hai đoạn mới.
2. Chạy cùng truy vấn với hai bộ lọc topic khác nhau.
3. Cập nhật một ID bằng upsert rồi kiểm tra nội dung mới.
4. Giải thích vì sao truy vấn không có đáp án vẫn có thể trả về top-k.
5. Không trộn embedding từ hai mô hình khác nhau trong cùng collection.

## Đọc schema: một hàng cần những trường nào?

IDs, documents, embeddings và metadatas là các danh sách song song: vị trí i trong mỗi danh sách thuộc cùng một bản ghi. ID nên ổn định để lần nạp sau có thể cập nhật đúng hàng. Source/page/topic giúp hiển thị nguồn và lọc trước khi đọc top-k.

Notebook dùng PersistentClient, nên dữ liệu còn sau khi kernel đóng. Dùng thư mục riêng chroma_data cho bài này; tránh gọi lệnh reset nếu chưa chắc muốn xóa database.

In [ ]:
# Kiểm tra rằng các batch có cùng số phần tử trước khi nạp
assert len(ids) == len(docs) == len(embeddings) == len(metadatas)
print("Bản ghi:", len(ids))
print("Số chiều vector:", embeddings.shape[1])
print("ID mẫu:", ids[0], "| metadata:", metadatas[0])

## Hiểu cấu trúc kết quả query

Khi gửi một query, nhiều trường trả về dưới dạng list lồng nhau: tầng ngoài là query, tầng trong là kết quả gần nhất. Vì vậy results["ids"][0] là ranking của query đầu tiên, còn results["ids"][1] là ranking của query thứ hai nếu gửi hai query.

Zip các trường cùng vị trí để tránh ghép nhầm document với metadata hoặc distance của hàng khác.

In [ ]:
# Đọc kết quả đã tạo ở phần truy vấn phía trên
first_ids = results["ids"][0]
first_docs = results["documents"][0]
first_meta = results["metadatas"][0]
first_distances = results["distances"][0]

for doc_id, document, metadata, distance in zip(
    first_ids, first_docs, first_meta, first_distances
):
    print(f"{doc_id} | distance={distance:.4f}")
    print(metadata)
    print(document)
    print("-" * 50)

## Phân biệt query với get

query xếp hạng theo gần vector, có thể kèm where để giới hạn topic/source. get lấy các bản ghi theo ID hoặc điều kiện metadata; nó không trả lời câu hỏi ngữ nghĩa. Dùng get để kiểm tra dữ liệu đã nạp và gỡ lỗi filter.

Nếu where lọc sai chính tả hoặc topic chưa được gắn cho bản ghi, truy vấn có thể rỗng. Hãy lấy metadata hiện có ra kiểm tra trước khi thay embedding model.

In [ ]:
# Kiểm tra dữ liệu lưu mà không xếp hạng theo vector
stored = collection.get(
    where={"topic": "retrieval"},
    include=["documents", "metadatas"]
)
print("Số bản ghi topic retrieval:", len(stored["ids"]))
for doc_id, metadata in zip(stored["ids"], stored["metadatas"]):
    print(doc_id, metadata)

## Bài tập kiểm tra hiểu biết

1. Bỏ where và chạy lại một query. Kết quả khác thế nào?
2. Đổi where sang topic không tồn tại. Vì sao danh sách rỗng?
3. Nạp lại cùng IDs bằng upsert. Count có tăng không? Vì sao?
4. Tạo một query vector có số chiều khác embedding đã lưu. Chroma cần báo lỗi gì?
5. Giải thích bằng lời vì sao distance nhỏ hơn biểu thị gần hơn với cosine metric.